<a href="https://colab.research.google.com/github/SKM1919/liftlens/blob/main/notebooks/03_criteo_uplift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LiftLens: Criteo Uplift (Hard tier)
Goal: scale uplift modelling to a large, imbalanced, real-world advertising experiment.

In [4]:
!mkdir -p data
!wget -q --show-progress -O data/criteo-uplift-v2.1.csv.gz \
  https://huggingface.co/datasets/criteo/criteo-uplift/resolve/main/criteo-research-uplift-v2.1.csv.gz

data/criteo-uplift- 100%[===================>] 297.00M  90.9MB/s    in 3.3s    


In [5]:
!pip install scikit-uplift xgboost -q

import pandas as pd
import numpy as np

SAMPLE_FRAC = 0.10
rng = np.random.default_rng(42)

chunks = []
for chunk in pd.read_csv('data/criteo-uplift-v2.1.csv.gz',
                         chunksize=1_000_000, dtype='float32'):
    chunks.append(chunk.sample(frac=SAMPLE_FRAC, random_state=rng))

df = pd.concat(chunks, ignore_index=True)

# Labels are 0/1, so store them compactly
for col in ['treatment', 'conversion', 'visit', 'exposure']:
    df[col] = df[col].astype('int8')

print("Shape:", df.shape)
print(f"Memory used: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()


Shape: (1397959, 16)
Memory used: 0.07 GB


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616364,10.059654,8.808347,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,24.334215,10.059654,8.214383,4.679882,10.280525,4.115453,-6.359691,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
2,21.900099,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
3,23.037542,10.059654,8.214383,4.679882,10.280525,4.115453,-4.595459,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0
4,26.223936,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0


In [6]:
df[[f'f{i}' for i in range(12)]].nunique().sort_values()

,0
f1,39
f11,67
f5,74
f4,132
f3,384
f9,977
f6,1101
f8,2273
f10,59342
f7,72835


In [7]:
print(f"Share in treatment group: {df['treatment'].mean() * 100:.1f}%")
print(f"Overall visit rate: {df['visit'].mean() * 100:.2f}%")
print(f"Overall conversion rate: {df['conversion'].mean() * 100:.3f}%")
print()

summary = df.groupby('treatment').agg(
    customers=('visit', 'size'),
    visit_rate=('visit', 'mean'),
    conversion_rate=('conversion', 'mean'),
    exposure_rate=('exposure', 'mean'),
    conversions=('conversion', 'sum'),
)
summary

Share in treatment group: 85.0%
Overall visit rate: 4.70%
Overall conversion rate: 0.289%



,customers,visit_rate,conversion_rate,exposure_rate,conversions
treatment,,,,,
0,209581,0.038615,0.001847,0.000000,387
1,1188378,0.048504,0.003071,0.036016,3650


In [8]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

treat = df[df['treatment'] == 1]
ctrl = df[df['treatment'] == 0]

for metric in ['visit', 'conversion']:
    x_t, n_t = treat[metric].sum(), len(treat)
    x_c, n_c = ctrl[metric].sum(), len(ctrl)
    z, p = proportions_ztest([x_t, x_c], [n_t, n_c])
    low, high = confint_proportions_2indep(x_t, n_t, x_c, n_c, method='wald')
    lift = x_t / n_t - x_c / n_c
    print(f"{metric:>10}: lift {lift * 100:.3f} pp  "
          f"(95% CI {low * 100:.3f} to {high * 100:.3f})  "
          f"relative +{lift / (x_c / n_c) * 100:.1f}%  p={p:.1e}")

     visit: lift 0.989 pp  (95% CI 0.898 to 1.080)  relative +25.6%  p=1.5e-86
conversion: lift 0.122 pp  (95% CI 0.102 to 0.143)  relative +66.3%  p=5.7e-22


### Findings
- **Design check:** 85% treatment / 15% control, as designed. Control exposure is exactly 0%, so the experiment is clean.
- **Only 3.6% of treated users actually saw an ad.** The measured lift is the intent-to-treat effect: the value of targeting someone, diluted by those never reached. This is the right number for targeting decisions.
- **Average effect:** visits +0.99 pp (95% CI 0.90 to 1.08, +26% relative); conversions +0.12 pp (95% CI 0.10 to 0.14, +66% relative).
- **Modelling target:** visit. The control group has only 387 conversions, too few to learn individual effects reliably, vs ~8,100 visits.
- **Never use `exposure` as a feature or for analysis:** it happens after assignment and depends on user behaviour, so it would leak information and break randomisation.